## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
from funcoes_escoragem import *

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [ ]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) <= CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,10377647659,4154573,2026-06-16,2026-06-16,1,E,2260.500000000,BLEND3_3,4154573,"{""pessoas"":[{""nome"":""KARLA DE OLIVEIRA BUENO"",...",...,33,1778785248777,"{""valor_aluguel"":""1700"",""imobiliaria_id"":35407...",2026-06-17 08:00:35+00:00,1781683232517756107,10377647659,E,2026-06-17 08:04:49.996000+00:00,2026-06-16 16:32:31+00:00,REPROVAR
1,36458218800,4159140,2026-06-16,2026-06-16,1,A,8083.000000000,BLEND3_3,4159140,"{""pessoas"":[{""nome"":""DIONE DE SOUZA VIANA"",""do...",...,34,1778785248777,"{""valor_aluguel"":""1500"",""imobiliaria_id"":6558,...",2026-06-16 18:00:37+00:00,1781632834370325077,36458218800,C,2026-06-16 18:03:53.506000+00:00,2026-06-16 11:52:06+00:00,APROVAR
2,70543608425,4167748,2026-06-08,2026-06-12,1,D,2192.000000000,BLEND3_3,4167748,"{""pessoas"":[{""nome"":""MAYARA SOARES DOS SANTOS""...",...,32,1778785248777,"{""valor_aluguel"":""670"",""imobiliaria_id"":14070,...",2026-06-13 08:00:40+00:00,1781337638615056544,70543608425,B,2026-06-13 08:05:45.545000+00:00,2026-06-12 16:14:50+00:00,APROVAR
3,11075243939,4172707,2026-06-08,2026-06-08,1,B,4178.500000000,BLEND3_3,4172707,"{""pessoas"":[{""nome"":""KEZIA DOS SANTOS DE SIQUE...",...,32,1778785248777,"{""valor_aluguel"":""1600"",""imobiliaria_id"":2303,...",2026-06-08 18:00:27+00:00,1780941620086066324,11075243939,B,2026-06-08 18:09:29.233000+00:00,2026-06-08 09:25:25+00:00,APROVAR
4,5382585350,4183036,2026-06-16,2026-06-16,1,NI,5685.500000000,BLEND3_3,4183036,"{""pessoas"":[{""nome"":""TALYTHA ALMEIDA RODRIGUES...",...,34,1778785248777,"{""valor_aluguel"":""1650"",""imobiliaria_id"":30556...",2026-06-16 18:00:37+00:00,1781632836177670654,05382585350,C,2026-06-16 18:03:54.416000+00:00,2026-06-16 14:36:26+00:00,DERIVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114036,15029620982,4333321,2026-07-04,2026-07-04,1,NI,0E-9,HVA4,4333321,"{""pessoas"":[{""nome"":""NAHUEL DAMIAN RIOS"",""docu...",...,33,1778785248777,"{""valor_aluguel"":2400,""matchmaking_on"":false,""...",2026-07-04 18:00:38+00:00,1783188037516106965,15029620982,E,2026-07-04 18:06:36.461000+00:00,2026-07-04 12:54:23+00:00,REPROVAR
114037,6698013056,4333371,2026-07-04,2026-07-04,1,NI,1849.500000000,BLEND3_3,4333371,"{""pessoas"":[{""nome"":""LUIZA PEREIRA DE CASTRO"",...",...,33,1778785248777,"{""valor_aluguel"":3500,""matchmaking_on"":false,""...",2026-07-04 18:00:38+00:00,1783188037603336971,06698013056,E,2026-07-04 18:06:36.478000+00:00,2026-07-04 13:34:59+00:00,REPROVAR
114038,5382219257,4333704,2026-07-05,2026-07-05,1,,0E-9,BLEND_4,4333704,"{""pessoas"":[{""nome"":"""",""documento"":""0538221925...",...,33,1778785248777,"{""principalDocument"":""05382219257"",""imobiliari...",2026-07-05 18:00:33+00:00,1783274432586373348,05382219257,E,2026-07-05 18:02:54.402000+00:00,2026-07-05 13:17:05+00:00,REPROVAR
114039,57976058720,4333747,2026-07-05,2026-07-05,1,E,1986.500000000,BLEND3_3,4333747,"{""pessoas"":[{""nome"":""MARIA DA CONCEICAO SILVA""...",...,37,1778785248777,"{""valor_aluguel"":1600,""matchmaking_on"":false,""...",2026-07-06 08:00:33+00:00,1783324831614452495,57976058720,C,2026-07-06 08:03:54.969000+00:00,2026-07-05 15:51:34+00:00,DERIVAR


In [6]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [7]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                97591
BLEND_REGRESSAO_2026     9053
HVA3                     4526
BVS_CUSTOM               1803
HVA4                      665
BLEND_4                   373
BVS_CUSTOM_V2              14
                           13
HFT1                        3
Name: count, dtype: int64

## Multiproponente

In [8]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    109611
2      4090
3       305
4        33
5         1
6         1
Name: count, dtype: Int64

In [9]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.961154
2    0.035864
3    0.002674
4    0.000289
5    0.000009
6    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [10]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [11]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

contrato_df = (
    pd.json_normalize(payload_parsed, sep="_")
    .add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.
)

pessoa_df = pd.json_normalize(
    payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0]),
    sep="_",
).add_prefix("pessoa_")

In [12]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [13]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_df = (
    pd.json_normalize(request_parsed.dropna(), sep="_")
    .add_prefix("request_")
)

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [14]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [15]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


### CredPago - Imóvel e Histórico de Valor

In [ ]:
# query = """
# WITH property_type AS (
#   SELECT
#     id AS contract_id,
#     CASE WHEN tp_imovel = 2 THEN 0 ELSE 1 END AS property_type,
#     CAST(id_cidade_ibge AS INT64) AS id_cidade_ibge,    --nova


#     TRIM(
#     REGEXP_REPLACE(
#         REGEXP_REPLACE(
#         UPPER(
#             REGEXP_REPLACE(
#             NORMALIZE(COALESCE(cidade, ''), NFD),
#             r'\pM', ''                      -- tira acento (marcas)
#             )
#         ),
#         r'[^A-Z0-9 ]', ' '                 -- tira lixo (agora já está tudo em maiúsculo)
#         ),
#         r'\s+', ' '                          -- colapsa espaços
#     )
#     ) AS contract_city_nm, --nova

#     CAST(id_imobiliaria AS INT64) AS agency_id,  --nova
#     DATE(criado_em) AS requested_at
#   FROM `loft-dl-fintech.bronze_credpago.imovel`
#   WHERE DATE(criado_em) >= DATE('2026-01-01')
#   QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY criado_em DESC) = 1
# ),

# rental_value AS (
#   SELECT
#     id_imovel AS contract_id,
#     CAST(vl_aluguel AS FLOAT64) AS rental_value,
#     DATE(data) AS requested_at
#   FROM `loft-dl-fintech.bronze_credpago.historico_valor`
#   WHERE DATE(data) >= DATE('2026-01-01')
#   QUALIFY ROW_NUMBER() OVER (PARTITION BY id_imovel ORDER BY data DESC) = 1
# )

# SELECT
#   COALESCE(p.contract_id, r.contract_id) AS contract_id,
#   GREATEST(p.requested_at, r.requested_at) AS requested_at,
#   p.property_type,
#   p.id_cidade_ibge,
#   p.contract_city_nm,
#   p.agency_id,
#   r.rental_value
# FROM property_type p
# FULL OUTER JOIN rental_value r
#   ON p.contract_id = r.contract_id;
# """

# credpago_imovel_df = pd.read_gbq(query, project_id=project_id)

In [ ]:
# credpago_imovel_df.head()

### Merge 

In [ ]:
# cp_final_df = pd.merge(credpago_clean, credpago_imovel_df.drop(columns=['requested_at']), on='contract_id', how='left')
# cp_final_df.head()

In [ ]:
# cols_ = ["property_type", "rental_value", "id_cidade_ibge", "agency_id"]

# pct_nulls = (
#     cp_final_df[cols_].isna().mean().mul(100).sort_values(ascending=False)
# )
# pct_nulls

In [ ]:
# pct_nulls_por_requested_at = (
#     cp_final_df.groupby("requested_at")[cols_]
#     .apply(lambda x: x.isna().mean().mul(100))
#     .sort_index()
# )

# pct_nulls_por_requested_at

## Escoragem Blend3

In [ ]:
rename_dict = {
    "message_scores_BVS_CUSTOM": "SCRCRDPNMGRLPFLGBCLFCREDPGV1",#
    "message_scores_HVA4": "SERASA_HVA4",
    "pessoa_idade": "age",
    "property_type": "property_type",  # peguei de fora da consulta realizada
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "rental_value": "rental_value",  # peguei de fora da consulta realizada
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [ ]:
vars_blend4 = ['SCRCRDPNMGRLPFLGBCLFCREDPGV1', 'SERASA_HVA4', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [ ]:
df_predict_blend3 = (cp_final_df.copy()).rename(columns=rename_dict)
# df_predict = df_predict[id_vars+vars_blend4]
df_predict_blend3["REGRA_BLEND_3"] = np.where(
    df_predict_blend3["SCRCRDPNMGRLPFLGBCLFCREDPGV1"] >= 435,
    "BLEND3",
    "E_BVS",
)

df_predict_blend3["SCORE_BVS"] = df_predict_blend3["SCRCRDPNMGRLPFLGBCLFCREDPGV1"]
df_predict_blend3["SCORE_SERASA"] = df_predict_blend3["SERASA_HVA4"]
df_predict_blend3["RENDA"] = df_predict_blend3["income"]

df_predict_blend3.head()

## Criação de Variáveis

In [ ]:
df_predict_blend3_vars = prepare_blend3_variables(df_predict_blend3)
df_predict_blend3_escorada = predict_blend3(df_predict_blend3_vars)

## Rating

In [ ]:
bvs = pd.to_numeric(df_predict_blend3_escorada["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(df_predict_blend3_escorada["predict_blend3_2_to_score"], errors="coerce")

conditions = [
    bvs.between(800, 1000),                                    # 800 ≤ BVS ≤ 1000  → A
    bvs.between(700, 799),                                     # 700 ≤ BVS < 800   → B
    bvs.between(620, 699),                                     # 620 ≤ BVS < 700   → C
    bvs.between(435, 619) & score.between(476, 1000),          # 435 ≤ BVS < 620, 476 ≤ pred ≤ 1000 → B
    bvs.between(435, 619) & score.between(399, 475),           # 435 ≤ BVS < 620, 399 ≤ pred < 476  → C
    bvs.between(435, 619) & score.between(387, 398),           # 435 ≤ BVS < 620, 387 ≤ pred < 399  → D
    bvs.between(435, 619) & score.between(0, 386),             # 435 ≤ BVS < 620, 0 ≤ pred < 387    → E
    bvs.between(0, 434),                                       # 0 ≤ BVS < 435    → E
]

choices = ["A", "B", "C", "B", "C", "D", "E", "E"]

df_predict_blend3_escorada["rating_manual_blend3"] = np.select(conditions, choices, default=None)

In [ ]:
bvs = pd.to_numeric(df_predict_blend3_escorada["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(df_predict_blend3_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    bvs.between(800, 1000),                                    # 800 ≤ BVS ≤ 1000  → A
    bvs.between(700, 799),                                     # 700 ≤ BVS < 800   → B
    bvs.between(620, 699),                                     # 620 ≤ BVS < 700   → C
    bvs.between(435, 619) & score.between(476, 1000),          # 435 ≤ BVS < 620, 476 ≤ pred ≤ 1000 → B
    bvs.between(435, 619) & score.between(399, 475),           # 435 ≤ BVS < 620, 399 ≤ pred < 476  → C
    bvs.between(435, 619) & score.between(387, 398),           # 435 ≤ BVS < 620, 387 ≤ pred < 399  → D
    bvs.between(435, 619) & score.between(0, 386),             # 435 ≤ BVS < 620, 0 ≤ pred < 387    → E
    bvs.between(0, 434),                                       # 0 ≤ BVS < 435    → E
]

choices = ["A", "B", "C", "B", "C", "D", "E", "E"]

df_predict_blend3_escorada["rating_json_blend3"] = np.select(conditions, choices, default=None)

## Salvar

In [ ]:
df_predict_blend3_escorada.to_csv(ANALYTICS_DIR/"df_predict_blend3.csv", index=False)